**Notebook configuration and imports**

In [0]:
# ============================================================
# DB_02_Supplier_Risk_Feature_Engineering
#
# Purpose:
# Build leakage-safe supplier-year features for the
# Supplier Risk Prediction model.
#
# Training design:
# Year T supplier performance -> Year T+1 risk outcome
#
# Data snapshot:
# 2026-07-31
# ============================================================

from datetime import date

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    DateType
)


# ------------------------------------------------------------
# Reproducible project configuration
# ------------------------------------------------------------

AS_OF_DATE = date(2026, 7, 31)

LATEST_COMPLETE_YEAR = 2025

SCORING_YEAR = 2026

MAX_FEATURE_YEAR_FOR_TRAINING = (
    LATEST_COMPLETE_YEAR - 1
)


# ------------------------------------------------------------
# Annualization factor for partial 2026 data
# ------------------------------------------------------------

days_in_scoring_year = (
    date(SCORING_YEAR, 12, 31)
    - date(SCORING_YEAR, 1, 1)
).days + 1

days_observed_scoring_year = (
    AS_OF_DATE
    - date(SCORING_YEAR, 1, 1)
).days + 1

SCORING_ANNUALIZATION_FACTOR = (
    days_in_scoring_year
    / days_observed_scoring_year
)


print("DB_02 configuration loaded.")
print("As-of date:", AS_OF_DATE)
print("Latest complete year:", LATEST_COMPLETE_YEAR)
print("Scoring year:", SCORING_YEAR)
print(
    "2026 annualization factor:",
    round(SCORING_ANNUALIZATION_FACTOR, 4)
)

DB_02 configuration loaded.
As-of date: 2026-07-31
Latest complete year: 2025
Scoring year: 2026
2026 annualization factor: 1.7217


**Load OneLake credentials securely**

This is intentionally repeated because DB_02 must work independently from DB_01.

In [0]:
# ============================================================
# Load Fabric OneLake credentials securely
# ============================================================

tenant_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-tenant-id"
)

client_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-id"
)

client_secret = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-secret"
)


print(
    "Fabric OneLake credentials loaded securely."
)

Fabric OneLake credentials loaded securely.


**Configure OneLake OAuth**

In [0]:
# ============================================================
# Configure Fabric OneLake OAuth
# ============================================================

spark.conf.set(
    "fs.azure.account.auth.type",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id",
    client_id
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret",
    client_secret
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint",
    (
        f"https://login.microsoftonline.com/"
        f"{tenant_id}/oauth2/token"
    )
)


print(
    "OneLake OAuth configuration applied."
)

OneLake OAuth configuration applied.


**Define Gold and ML feature paths**

In [0]:
# ============================================================
# Fabric Gold and ML paths
# ============================================================

FACT_SUPPLIER_PERFORMANCE_PATH = (
    "abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Tables/fact_supplier_performance"
)


# ------------------------------------------------------------
# Derive the Gold Lakehouse root automatically
# ------------------------------------------------------------

GOLD_LAKEHOUSE_ROOT = (
    FACT_SUPPLIER_PERFORMANCE_PATH
    .rsplit(
        "/Tables/",
        1
    )[0]
)


DIM_SUPPLIER_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/dim_supplier"
)


# ------------------------------------------------------------
# Intermediate ML feature store
#
# We deliberately use /Files rather than Gold /Tables.
#
# Final ML predictions will later be written as curated
# Gold tables by DB_07.
# ------------------------------------------------------------

SUPPLIER_RISK_ML_ROOT = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Files/ml/supplier_risk"
)

TRAINING_FEATURES_PATH = (
    f"{SUPPLIER_RISK_ML_ROOT}/"
    f"training_features"
)

SCORING_FEATURES_PATH = (
    f"{SUPPLIER_RISK_ML_ROOT}/"
    f"scoring_features"
)

LABEL_THRESHOLDS_PATH = (
    f"{SUPPLIER_RISK_ML_ROOT}/"
    f"label_thresholds"
)


print("Gold Lakehouse root:")
print(GOLD_LAKEHOUSE_ROOT)

print("\nSupplier dimension:")
print(DIM_SUPPLIER_PATH)

print("\nTraining features:")
print(TRAINING_FEATURES_PATH)

print("\nScoring features:")
print(SCORING_FEATURES_PATH)

Gold Lakehouse root:
abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse

Supplier dimension:
abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Tables/dim_supplier

Training features:
abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/supplier_risk/training_features

Scoring features:
abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/supplier_risk/scoring_features


**Read the Gold source tables**

In [0]:
# ============================================================
# Read Gold source tables
# ============================================================

fact_supplier_performance_df = (
    spark.read
    .format("delta")
    .load(
        FACT_SUPPLIER_PERFORMANCE_PATH
    )
)


dim_supplier_df = (
    spark.read
    .format("delta")
    .load(
        DIM_SUPPLIER_PATH
    )
)


supplier_performance_count = (
    fact_supplier_performance_df.count()
)

supplier_dimension_count = (
    dim_supplier_df.count()
)


print(
    "fact_supplier_performance rows:",
    f"{supplier_performance_count:,}"
)

print(
    "dim_supplier rows:",
    f"{supplier_dimension_count:,}"
)

fact_supplier_performance rows: 1,869
dim_supplier rows: 500


**Validate required Gold columns**

In [0]:
# ============================================================
# Validate source schemas
# ============================================================

required_fact_columns = [
    "SupplierKey",
    "PerformanceYear",

    # Spend / compliance
    "EligibleSpendEUR",
    "ContractCompliancePct",
    "MaverickSpendPct",

    # Delivery
    "SupplierOTDPct",
    "FullyReceivedPOItemCount",
    "OnTimeFullyReceivedPOItemCount",
    "OverdueOpenDeliveryCount",

    # Invoice / quality
    "SupplierQualityIndexPct",
    "InvoiceCount",
    "DisputedInvoiceCount",
    "DuplicateInvoicePct",
    "ThreeWayMatchPct",
    "InvoiceExceptionPct"
]


required_supplier_columns = [
    "SupplierKey",
    "SupplierID",
    "SupplierName",
    "SupplierType",
    "Country",
    "Region",
    "PreferredSupplier",
    "StrategicSupplier",
    "ESGRating",
    "FinancialRiskScore",
    "Status"
]


missing_fact_columns = sorted(
    set(required_fact_columns)
    - set(fact_supplier_performance_df.columns)
)

missing_supplier_columns = sorted(
    set(required_supplier_columns)
    - set(dim_supplier_df.columns)
)


if missing_fact_columns:
    raise ValueError(
        "Missing required columns from "
        "fact_supplier_performance: "
        + ", ".join(missing_fact_columns)
    )


if missing_supplier_columns:
    raise ValueError(
        "Missing required columns from dim_supplier: "
        + ", ".join(missing_supplier_columns)
    )


print("Source schema validation PASSED.")

print(
    "Required fact columns validated:",
    len(required_fact_columns)
)

print(
    "Required supplier columns validated:",
    len(required_supplier_columns)
)

Source schema validation PASSED.
Required fact columns validated: 15
Required supplier columns validated: 11


**Build the base supplier-year modeling dataset**

In [0]:
# ============================================================
# Build supplier-year modeling base
# ============================================================

f = fact_supplier_performance_df.alias("f")
d = dim_supplier_df.alias("d")


supplier_year_base_df = (
    f

    .join(
        d,
        F.col("f.SupplierKey")
        == F.col("d.SupplierKey"),
        "left"
    )

    .select(
        # ----------------------------------------------------
        # Grain
        # ----------------------------------------------------

        F.col("f.SupplierKey")
        .alias("SupplierKey"),

        F.col("d.SupplierID")
        .alias("SupplierID"),

        F.col("d.SupplierName")
        .alias("SupplierName"),

        F.col("f.PerformanceYear")
        .cast("int")
        .alias("PerformanceYear"),

        # ----------------------------------------------------
        # Supplier attributes
        # ----------------------------------------------------

        F.col("d.SupplierType")
        .alias("SupplierType"),

        F.col("d.Country")
        .alias("Country"),

        F.col("d.Region")
        .alias("Region"),

        F.col("d.PreferredSupplier")
        .cast("int")
        .alias("PreferredSupplierFlag"),

        F.col("d.StrategicSupplier")
        .cast("int")
        .alias("StrategicSupplierFlag"),

        # ESG Rating is categorical, e.g. A / B / C.
        # Keep it as STRING. It will be encoded in DB_03.
        F.col("d.ESGRating")
        .cast("string")
        .alias("ESGRating"),

        # Financial Risk Score is genuinely numeric.
        F.col("d.FinancialRiskScore")
        .cast("double")
        .alias("FinancialRiskScore"),

        F.col("d.Status")
        .alias("SupplierStatus"),

        # ----------------------------------------------------
        # Spend / compliance
        # ----------------------------------------------------

        F.col("f.EligibleSpendEUR")
        .cast("double")
        .alias("EligibleSpendEUR"),

        F.col("f.ContractCompliancePct")
        .cast("double")
        .alias("ContractCompliancePct"),

        F.col("f.MaverickSpendPct")
        .cast("double")
        .alias("MaverickSpendPct"),

        # ----------------------------------------------------
        # Delivery
        # ----------------------------------------------------

        F.col("f.SupplierOTDPct")
        .cast("double")
        .alias("SupplierOTDPct"),

        F.col("f.FullyReceivedPOItemCount")
        .cast("double")
        .alias("FullyReceivedPOItemCount"),

        F.col("f.OnTimeFullyReceivedPOItemCount")
        .cast("double")
        .alias("OnTimeFullyReceivedPOItemCount"),

        F.col("f.OverdueOpenDeliveryCount")
        .cast("double")
        .alias("OverdueOpenDeliveryCount"),

        # ----------------------------------------------------
        # Invoice / supplier quality
        # ----------------------------------------------------

        F.col("f.SupplierQualityIndexPct")
        .cast("double")
        .alias("SupplierQualityIndexPct"),

        F.col("f.InvoiceCount")
        .cast("double")
        .alias("InvoiceCount"),

        F.col("f.DisputedInvoiceCount")
        .cast("double")
        .alias("DisputedInvoiceCount"),

        F.col("f.DuplicateInvoicePct")
        .cast("double")
        .alias("DuplicateInvoicePct"),

        F.col("f.ThreeWayMatchPct")
        .cast("double")
        .alias("ThreeWayMatchPct"),

        F.col("f.InvoiceExceptionPct")
        .cast("double")
        .alias("InvoiceExceptionPct")
    )

    # ========================================================
    # Derived ML features
    # ========================================================

    # --------------------------------------------------------
    # Invoice dispute rate
    # --------------------------------------------------------

    .withColumn(
        "InvoiceDisputePct",
        F.when(
            F.col("InvoiceCount") > 0,
            (
                F.col("DisputedInvoiceCount")
                / F.col("InvoiceCount")
            ) * 100.0
        )
    )

    # --------------------------------------------------------
    # Late fully received deliveries
    # --------------------------------------------------------

    .withColumn(
        "LateFullyReceivedPOItemCount",
        F.greatest(
            F.col("FullyReceivedPOItemCount")
            - F.col("OnTimeFullyReceivedPOItemCount"),
            F.lit(0.0)
        )
    )

    .withColumn(
        "LateFullyReceivedPct",
        F.when(
            F.col("FullyReceivedPOItemCount") > 0,
            (
                F.col("LateFullyReceivedPOItemCount")
                / F.col("FullyReceivedPOItemCount")
            ) * 100.0
        )
    )

    # --------------------------------------------------------
    # Open overdue delivery exposure
    # --------------------------------------------------------

    .withColumn(
        "OverdueOpenDeliveryExposurePct",
        F.when(
            (
                F.col("FullyReceivedPOItemCount")
                + F.col("OverdueOpenDeliveryCount")
            ) > 0,
            (
                F.col("OverdueOpenDeliveryCount")
                /
                (
                    F.col("FullyReceivedPOItemCount")
                    + F.col("OverdueOpenDeliveryCount")
                )
            ) * 100.0
        )
    )
)


print(
    "Supplier-year base rows:",
    f"{supplier_year_base_df.count():,}"
)


display(
    supplier_year_base_df
    .orderBy(
        "SupplierID",
        "PerformanceYear"
    )
    .limit(20)
)

Supplier-year base rows: 1,869


SupplierKey,SupplierID,SupplierName,PerformanceYear,SupplierType,Country,Region,PreferredSupplierFlag,StrategicSupplierFlag,ESGRating,FinancialRiskScore,SupplierStatus,EligibleSpendEUR,ContractCompliancePct,MaverickSpendPct,SupplierOTDPct,FullyReceivedPOItemCount,OnTimeFullyReceivedPOItemCount,OverdueOpenDeliveryCount,SupplierQualityIndexPct,InvoiceCount,DisputedInvoiceCount,DuplicateInvoicePct,ThreeWayMatchPct,InvoiceExceptionPct,InvoiceDisputePct,LateFullyReceivedPOItemCount,LateFullyReceivedPct,OverdueOpenDeliveryExposurePct
-2251272880525291685,SUP000001,Meridian Integrated Distribution LLC,2022,Manufacturer,United States,Americas,1,0,D,47.0,Active,7309634.54,1.39,98.61,75.0,40.0,30.0,1.0,100.0,9.0,0.0,0.0,100.0,0.0,0.0,10.0,25.0,2.4390243902439024
-2251272880525291685,SUP000001,Meridian Integrated Distribution LLC,2023,Manufacturer,United States,Americas,1,0,D,47.0,Active,961814.87,2.95,97.05,80.77,26.0,21.0,9.0,92.31,13.0,1.0,7.69,86.05,13.95,7.6923076923076925,5.0,19.230769230769234,25.71428571428571
-2251272880525291685,SUP000001,Meridian Integrated Distribution LLC,2024,Manufacturer,United States,Americas,1,0,D,47.0,Active,736990.34,31.38,68.62,72.41,29.0,21.0,4.0,100.0,10.0,0.0,0.0,89.66,10.34,0.0,8.0,27.586206896551722,12.121212121212121
-2251272880525291685,SUP000001,Meridian Integrated Distribution LLC,2025,Manufacturer,United States,Americas,1,0,D,47.0,Active,1374428.65,5.92,94.08,88.57,35.0,31.0,0.0,100.0,10.0,0.0,0.0,100.0,0.0,0.0,4.0,11.428571428571429,0.0
-2251272880525291685,SUP000001,Meridian Integrated Distribution LLC,2026,Manufacturer,United States,Americas,1,0,D,47.0,Active,2.533565349E7,77.83,22.17,81.48,54.0,44.0,28.0,97.37,38.0,1.0,0.0,92.86,7.14,2.631578947368421,10.0,18.51851851851852,34.146341463414636
-2044949693076992319,SUP000002,Redwood Integrated Services Ltd.,2023,Service Provider,Spain,EMEA,0,0,D,21.0,Active,31004.97,0.0,100.0,100.0,5.0,5.0,0.0,100.0,1.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0
-2044949693076992319,SUP000002,Redwood Integrated Services Ltd.,2026,Service Provider,Spain,EMEA,0,0,D,21.0,Active,92543.53,0.0,100.0,80.0,5.0,4.0,1.0,100.0,2.0,0.0,0.0,16.67,83.33,0.0,1.0,20.0,16.666666666666664
5964456763634693253,SUP000003,Pioneer Advanced Materials Group,2024,Utility Provider,Malaysia,APAC,0,0,C,17.0,Active,33763.55,0.0,100.0,80.0,5.0,4.0,0.0,100.0,1.0,0.0,0.0,100.0,0.0,0.0,1.0,20.0,0.0
5964456763634693253,SUP000003,Pioneer Advanced Materials Group,2025,Utility Provider,Malaysia,APAC,0,0,C,17.0,Active,45187.69,0.0,100.0,80.0,5.0,4.0,0.0,100.0,1.0,0.0,0.0,100.0,0.0,0.0,1.0,20.0,0.0
6335603648337764032,SUP000004,Frontier International Automation AG,2022,Logistics Provider,Switzerland,EMEA,0,0,C,56.0,Active,788999.22,78.3,21.7,74.14,58.0,43.0,3.0,100.0,12.0,0.0,0.0,98.0,2.0,0.0,15.0,25.862068965517242,4.918032786885246


**Validate the supplier dimension join**

We should have zero unresolved suppliers

In [0]:
# ============================================================
# Validate supplier dimension join
# ============================================================

unmatched_supplier_rows = (
    supplier_year_base_df

    .filter(
        F.col("SupplierID").isNull()
    )

    .count()
)


duplicate_supplier_year_rows = (
    supplier_year_base_df

    .groupBy(
        "SupplierID",
        "PerformanceYear"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


print(
    "Unmatched supplier rows:",
    unmatched_supplier_rows
)

print(
    "Duplicate Supplier-Year grains:",
    duplicate_supplier_year_rows
)


if unmatched_supplier_rows != 0:
    raise ValueError(
        "Supplier dimension join validation failed."
    )


if duplicate_supplier_year_rows != 0:
    raise ValueError(
        "Supplier-Year grain validation failed."
    )


print(
    "Supplier dimension join validation PASSED."
)

Unmatched supplier rows: 0
Duplicate Supplier-Year grains: 0
Supplier dimension join validation PASSED.


**Annualize the partial 2026 activity**

In [0]:
# ============================================================
# Make partial-year activity comparable with full years
# ============================================================

supplier_year_comparable_df = (
    supplier_year_base_df

    .withColumn(
        "IsPartialYearFlag",
        F.when(
            F.col("PerformanceYear")
            == SCORING_YEAR,
            F.lit(1)
        )
        .otherwise(
            F.lit(0)
        )
    )

    .withColumn(
        "AnnualizationFactor",
        F.when(
            F.col("PerformanceYear")
            == SCORING_YEAR,
            F.lit(
                SCORING_ANNUALIZATION_FACTOR
            )
        )
        .otherwise(
            F.lit(1.0)
        )
    )

    .withColumn(
        "AnnualizedEligibleSpendEUR",
        F.col("EligibleSpendEUR")
        * F.col("AnnualizationFactor")
    )

    .withColumn(
        "AnnualizedFullyReceivedPOItemCount",
        F.col("FullyReceivedPOItemCount")
        * F.col("AnnualizationFactor")
    )

    .withColumn(
        "AnnualizedInvoiceCount",
        F.col("InvoiceCount")
        * F.col("AnnualizationFactor")
    )

    .withColumn(
        "AnnualizedDisputedInvoiceCount",
        F.col("DisputedInvoiceCount")
        * F.col("AnnualizationFactor")
    )

    .withColumn(
        "AnnualizedOverdueOpenDeliveryCount",
        F.col("OverdueOpenDeliveryCount")
        * F.col("AnnualizationFactor")
    )
)


display(
    supplier_year_comparable_df

    .filter(
        F.col("PerformanceYear")
        == SCORING_YEAR
    )

    .select(
        "SupplierID",
        "PerformanceYear",
        "EligibleSpendEUR",
        "AnnualizationFactor",
        "AnnualizedEligibleSpendEUR",
        "FullyReceivedPOItemCount",
        "AnnualizedFullyReceivedPOItemCount",
        "InvoiceCount",
        "AnnualizedInvoiceCount"
    )

    .orderBy(
        F.desc(
            "EligibleSpendEUR"
        )
    )

    .limit(20)
)

SupplierID,PerformanceYear,EligibleSpendEUR,AnnualizationFactor,AnnualizedEligibleSpendEUR,FullyReceivedPOItemCount,AnnualizedFullyReceivedPOItemCount,InvoiceCount,AnnualizedInvoiceCount
SUP000058,2026,1.6440467472E8,1.721698113207547,2.830552182679245E8,4.0,6.886792452830188,18.0,30.990566037735846
SUP000404,2026,1.2344658151E8,1.721698113207547,2.1253774646768868E8,85.0,146.3443396226415,44.0,75.75471698113208
SUP000496,2026,8.684655516E7,1.721698113207547,1.4952355015754715E8,40.0,68.86792452830188,27.0,46.48584905660377
SUP000312,2026,7.930130726E7,1.721698113207547,1.3653291108443397E8,65.0,111.91037735849056,31.0,53.37264150943396
SUP000440,2026,7.854115607E7,1.721698113207547,1.3522416021485847E8,108.0,185.94339622641508,53.0,91.25
SUP000142,2026,7.158210409E7,1.721698113207547,1.2324277355117925E8,12.0,20.660377358490564,11.0,18.93867924528302
SUP000419,2026,7.111722212E7,1.721698113207547,1.2244238714056604E8,187.0,321.9575471698113,82.0,141.17924528301887
SUP000228,2026,6.534570429E7,1.721698113207547,1.125055757823113E8,147.0,253.0896226415094,77.0,132.57075471698113
SUP000084,2026,6.324376883E7,1.721698113207547,1.0888667746674527E8,35.0,60.25943396226415,20.0,34.43396226415094
SUP000353,2026,6.288859455E7,1.721698113207547,1.0827517457900943E8,98.0,168.72641509433961,46.0,79.19811320754717


**Engineer historical and trend features**

In [0]:
# ============================================================
# Historical supplier risk feature engineering
# ============================================================

supplier_history_window = (
    Window
    .partitionBy("SupplierID")
    .orderBy("PerformanceYear")
)


supplier_rolling_3y_window = (
    Window
    .partitionBy("SupplierID")
    .orderBy(
        F.col("PerformanceYear").cast("long")
    )
    .rangeBetween(
        -2,
        0
    )
)


year_total_window = (
    Window
    .partitionBy("PerformanceYear")
)


supplier_feature_df = (
    supplier_year_comparable_df

    # --------------------------------------------------------
    # Previous observed year
    # --------------------------------------------------------

    .withColumn(
        "PriorObservedYear",
        F.lag(
            "PerformanceYear",
            1
        ).over(
            supplier_history_window
        )
    )

    .withColumn(
        "_LagSpendEUR",
        F.lag(
            "AnnualizedEligibleSpendEUR",
            1
        ).over(
            supplier_history_window
        )
    )

    .withColumn(
        "_LagOTDPct",
        F.lag(
            "SupplierOTDPct",
            1
        ).over(
            supplier_history_window
        )
    )

    .withColumn(
        "_LagInvoiceDisputePct",
        F.lag(
            "InvoiceDisputePct",
            1
        ).over(
            supplier_history_window
        )
    )

    .withColumn(
        "_LagMaverickSpendPct",
        F.lag(
            "MaverickSpendPct",
            1
        ).over(
            supplier_history_window
        )
    )

    # --------------------------------------------------------
    # Only use lag where previous calendar year exists
    # --------------------------------------------------------

    .withColumn(
        "PriorYearSpendEUR",
        F.when(
            F.col("PriorObservedYear")
            == F.col("PerformanceYear") - 1,
            F.col("_LagSpendEUR")
        )
    )

    .withColumn(
        "PriorYearOTDPct",
        F.when(
            F.col("PriorObservedYear")
            == F.col("PerformanceYear") - 1,
            F.col("_LagOTDPct")
        )
    )

    .withColumn(
        "PriorYearInvoiceDisputePct",
        F.when(
            F.col("PriorObservedYear")
            == F.col("PerformanceYear") - 1,
            F.col("_LagInvoiceDisputePct")
        )
    )

    .withColumn(
        "PriorYearMaverickSpendPct",
        F.when(
            F.col("PriorObservedYear")
            == F.col("PerformanceYear") - 1,
            F.col("_LagMaverickSpendPct")
        )
    )

    # --------------------------------------------------------
    # Year-over-year behavioral changes
    # --------------------------------------------------------

    .withColumn(
        "YoYSpendChangePct",
        F.when(
            F.col("PriorYearSpendEUR") > 0,
            (
                (
                    F.col("AnnualizedEligibleSpendEUR")
                    - F.col("PriorYearSpendEUR")
                )
                / F.col("PriorYearSpendEUR")
            ) * 100.0
        )
    )

    .withColumn(
        "YoYOTDChangePp",
        F.when(
            F.col("PriorYearOTDPct").isNotNull(),
            F.col("SupplierOTDPct")
            - F.col("PriorYearOTDPct")
        )
    )

    .withColumn(
        "YoYInvoiceDisputeChangePp",
        F.when(
            F.col(
                "PriorYearInvoiceDisputePct"
            ).isNotNull(),
            F.col("InvoiceDisputePct")
            - F.col(
                "PriorYearInvoiceDisputePct"
            )
        )
    )

    .withColumn(
        "YoYMaverickSpendChangePp",
        F.when(
            F.col(
                "PriorYearMaverickSpendPct"
            ).isNotNull(),
            F.col("MaverickSpendPct")
            - F.col(
                "PriorYearMaverickSpendPct"
            )
        )
    )

    # --------------------------------------------------------
    # Rolling 3-year spend behavior
    # --------------------------------------------------------

    .withColumn(
        "Rolling3YObservationCount",
        F.count(
            "AnnualizedEligibleSpendEUR"
        ).over(
            supplier_rolling_3y_window
        )
    )

    .withColumn(
        "Rolling3YAverageSpendEUR",
        F.avg(
            "AnnualizedEligibleSpendEUR"
        ).over(
            supplier_rolling_3y_window
        )
    )

    .withColumn(
        "Rolling3YSpendStdDevEUR",
        F.stddev_samp(
            "AnnualizedEligibleSpendEUR"
        ).over(
            supplier_rolling_3y_window
        )
    )

    .withColumn(
        "SpendVolatilityPct",
        F.when(
            (
                F.col(
                    "Rolling3YObservationCount"
                ) >= 2
            )
            &
            (
                F.col(
                    "Rolling3YAverageSpendEUR"
                ) > 0
            ),
            (
                F.col(
                    "Rolling3YSpendStdDevEUR"
                )
                /
                F.col(
                    "Rolling3YAverageSpendEUR"
                )
            ) * 100.0
        )
    )

    # --------------------------------------------------------
    # Relative supplier importance
    # --------------------------------------------------------

    .withColumn(
        "_TotalYearSpendEUR",
        F.sum(
            "AnnualizedEligibleSpendEUR"
        ).over(
            year_total_window
        )
    )

    .withColumn(
        "SupplierSpendSharePct",
        F.when(
            F.col("_TotalYearSpendEUR") > 0,
            (
                F.col(
                    "AnnualizedEligibleSpendEUR"
                )
                /
                F.col(
                    "_TotalYearSpendEUR"
                )
            ) * 100.0
        )
    )

    .withColumn(
        "LogAnnualizedEligibleSpendEUR",
        F.log1p(
            F.greatest(
                F.col(
                    "AnnualizedEligibleSpendEUR"
                ),
                F.lit(0.0)
            )
        )
    )

    # --------------------------------------------------------
    # Risk observation coverage
    # --------------------------------------------------------

    .withColumn(
        "RiskMetricObservedCount",
        (
            F.when(
                F.col(
                    "SupplierOTDPct"
                ).isNotNull(),
                1
            ).otherwise(0)

            +

            F.when(
                F.col(
                    "OverdueOpenDeliveryExposurePct"
                ).isNotNull(),
                1
            ).otherwise(0)

            +

            F.when(
                F.col(
                    "InvoiceDisputePct"
                ).isNotNull(),
                1
            ).otherwise(0)

            +

            F.when(
                F.col(
                    "InvoiceExceptionPct"
                ).isNotNull(),
                1
            ).otherwise(0)
        )
    )

    .withColumn(
        "RiskMetricCoveragePct",
        (
            F.col("RiskMetricObservedCount")
            / F.lit(4.0)
        ) * 100.0
    )

    # --------------------------------------------------------
    # Remove internal columns
    # --------------------------------------------------------

    .drop(
        "_LagSpendEUR",
        "_LagOTDPct",
        "_LagInvoiceDisputePct",
        "_LagMaverickSpendPct",
        "_TotalYearSpendEUR"
    )
)


print(
    "Feature-engineered rows:",
    f"{supplier_feature_df.count():,}"
)

Feature-engineered rows: 1,869


**Inspect the engineered features**

In [0]:
# ============================================================
# Inspect engineered supplier features
# ============================================================

display(
    supplier_feature_df

    .select(
        "SupplierID",
        "SupplierName",
        "PerformanceYear",

        "AnnualizedEligibleSpendEUR",
        "PriorYearSpendEUR",
        "YoYSpendChangePct",
        "Rolling3YAverageSpendEUR",
        "SpendVolatilityPct",

        "SupplierOTDPct",
        "LateFullyReceivedPct",
        "OverdueOpenDeliveryExposurePct",
        "YoYOTDChangePp",

        "InvoiceDisputePct",
        "YoYInvoiceDisputeChangePp",
        "InvoiceExceptionPct",

        "FinancialRiskScore",
        "ESGRating",
        "SupplierSpendSharePct"
    )

    .orderBy(
        "SupplierID",
        "PerformanceYear"
    )

    .limit(50)
)

SupplierID,SupplierName,PerformanceYear,AnnualizedEligibleSpendEUR,PriorYearSpendEUR,YoYSpendChangePct,Rolling3YAverageSpendEUR,SpendVolatilityPct,SupplierOTDPct,LateFullyReceivedPct,OverdueOpenDeliveryExposurePct,YoYOTDChangePp,InvoiceDisputePct,YoYInvoiceDisputeChangePp,InvoiceExceptionPct,FinancialRiskScore,ESGRating,SupplierSpendSharePct
SUP000001,Meridian Integrated Distribution LLC,2022,7309634.54,null,null,7309634.54,null,75.0,25.0,2.4390243902439024,null,0.0,null,0.0,47.0,D,0.5239469007474583
SUP000001,Meridian Integrated Distribution LLC,2023,961814.87,7309634.54,-86.84182000157836,4135724.705,108.53203862867736,80.77,19.230769230769234,25.71428571428571,5.769999999999996,7.6923076923076925,7.6923076923076925,13.95,47.0,D,0.09730043560416021
SUP000001,Meridian Integrated Distribution LLC,2024,736990.34,961814.87,-23.37503162121002,3002813.25,124.2671434391871,72.41,27.586206896551722,12.121212121212121,-8.36,0.0,-7.6923076923076925,10.34,47.0,D,0.06430644407965301
SUP000001,Meridian Integrated Distribution LLC,2025,1374428.65,736990.34,86.49208482162737,1024411.2866666666,31.559249110658733,88.57,11.428571428571429,0.0,16.159999999999997,0.0,0.0,0.0,47.0,D,0.08009664607139283
SUP000001,Meridian Integrated Distribution LLC,2026,4.36203468106132E7,1374428.65,3073.7076210258865,1.5243921933537735E7,161.2234150131504,81.48,18.51851851851852,34.146341463414636,-7.089999999999989,2.631578947368421,2.631578947368421,7.14,47.0,D,1.1425136740771058
SUP000002,Redwood Integrated Services Ltd.,2023,31004.97,null,null,31004.97,null,100.0,0.0,0.0,null,0.0,null,0.0,21.0,D,0.003136567317673015
SUP000002,Redwood Integrated Services Ltd.,2026,159332.02099056603,null,null,159332.02099056603,null,80.0,20.0,16.666666666666664,null,0.0,null,83.33,21.0,D,0.004173259178576052
SUP000003,Pioneer Advanced Materials Group,2024,33763.55,null,null,33763.55,null,80.0,20.0,0.0,null,0.0,null,0.0,17.0,C,0.0029460546796387706
SUP000003,Pioneer Advanced Materials Group,2025,45187.69,33763.55,33.835719288996565,39475.62,20.463483190953006,80.0,20.0,0.0,0.0,0.0,0.0,0.0,17.0,C,0.00263337235637064
SUP000004,Frontier International Automation AG,2022,788999.22,null,null,788999.22,null,74.14,25.862068965517242,4.918032786885246,null,0.0,null,2.0,56.0,C,0.05655463262205199


**Create the future supplier-risk outcomes**

This is the key leakage-control step.

The current year does not determine its own target.

Instead:

2022 features ← 2023 actual outcome
2023 features ← 2024 actual outcome
2024 features ← 2025 actual outcome

In [0]:
# ============================================================
# Build future-year supplier risk outcomes
# ============================================================

future_outcome_df = (
    supplier_year_base_df

    .filter(
        F.col("PerformanceYear")
        <= LATEST_COMPLETE_YEAR
    )

    .select(
        F.col("SupplierID"),

        (
            F.col("PerformanceYear") - 1
        ).alias("FeatureYear"),

        F.col("PerformanceYear")
        .alias("FutureOutcomeYear"),

        F.col("SupplierOTDPct")
        .alias(
            "FutureSupplierOTDPct"
        ),

        F.col(
            "OverdueOpenDeliveryExposurePct"
        )
        .alias(
            "FutureOverdueOpenDeliveryExposurePct"
        ),

        F.col("InvoiceDisputePct")
        .alias(
            "FutureInvoiceDisputePct"
        ),

        F.col("InvoiceExceptionPct")
        .alias(
            "FutureInvoiceExceptionPct"
        )
    )
)


training_label_base_df = (
    supplier_feature_df

    .filter(
        F.col("PerformanceYear")
        <= MAX_FEATURE_YEAR_FOR_TRAINING
    )

    .alias("features")

    .join(
        future_outcome_df.alias("future"),

        (
            F.col("features.SupplierID")
            == F.col("future.SupplierID")
        )
        &
        (
            F.col("features.PerformanceYear")
            == F.col("future.FeatureYear")
        ),

        "inner"
    )

    .select(
        "features.*",

        F.col(
            "future.FutureOutcomeYear"
        ),

        F.col(
            "future.FutureSupplierOTDPct"
        ),

        F.col(
            "future.FutureOverdueOpenDeliveryExposurePct"
        ),

        F.col(
            "future.FutureInvoiceDisputePct"
        ),

        F.col(
            "future.FutureInvoiceExceptionPct"
        )
    )
)


print(
    "Candidate training rows:",
    f"{training_label_base_df.count():,}"
)

Candidate training rows: 1,019


**Define future-risk thresholds from training outcomes**

Instead of inventing arbitrary thresholds for our synthetic data, we define adverse future performance relative to the supplier population.

For the training population:

- low future OTD = bottom quartile
- high overdue delivery = top quartile
- high invoice dispute = top quartile
- high invoice exception = top quartile

This makes the target data-driven and reproducible.

In [0]:
# ============================================================
# Derive supplier risk label thresholds
# ============================================================

def get_quantile(
    dataframe,
    column_name,
    probability
):
    result = dataframe.approxQuantile(
        column_name,
        [probability],
        0.01
    )

    if not result:
        raise ValueError(
            f"Unable to calculate quantile "
            f"for {column_name}."
        )

    return float(
        result[0]
    )


OTD_LOW_THRESHOLD = get_quantile(
    training_label_base_df,
    "FutureSupplierOTDPct",
    0.25
)


OVERDUE_EXPOSURE_HIGH_THRESHOLD = get_quantile(
    training_label_base_df,
    "FutureOverdueOpenDeliveryExposurePct",
    0.75
)


DISPUTE_HIGH_THRESHOLD = get_quantile(
    training_label_base_df,
    "FutureInvoiceDisputePct",
    0.75
)


EXCEPTION_HIGH_THRESHOLD = get_quantile(
    training_label_base_df,
    "FutureInvoiceExceptionPct",
    0.75
)


print("Future supplier risk thresholds")

print(
    "Low OTD threshold:",
    round(
        OTD_LOW_THRESHOLD,
        4
    )
)

print(
    "High overdue exposure threshold:",
    round(
        OVERDUE_EXPOSURE_HIGH_THRESHOLD,
        4
    )
)

print(
    "High invoice dispute threshold:",
    round(
        DISPUTE_HIGH_THRESHOLD,
        4
    )
)

print(
    "High invoice exception threshold:",
    round(
        EXCEPTION_HIGH_THRESHOLD,
        4
    )
)

Future supplier risk thresholds
Low OTD threshold: 82.54
High overdue exposure threshold: 5.9524
High invoice dispute threshold: 3.7037
High invoice exception threshold: 9.68


**Create the supervised high-risk target**

A supplier is classified as _HighRiskNextYearFlag_ = 1 when at least two future adverse conditions occur.

In [0]:
# ============================================================
# Create future supplier risk target
# ============================================================

training_labeled_df = (
    training_label_base_df

    # --------------------------------------------------------
    # Future metric coverage
    # --------------------------------------------------------

    .withColumn(
        "FutureRiskObservedMetricCount",
        (
            F.when(
                F.col(
                    "FutureSupplierOTDPct"
                ).isNotNull(),
                1
            ).otherwise(0)

            +

            F.when(
                F.col(
                    "FutureOverdueOpenDeliveryExposurePct"
                ).isNotNull(),
                1
            ).otherwise(0)

            +

            F.when(
                F.col(
                    "FutureInvoiceDisputePct"
                ).isNotNull(),
                1
            ).otherwise(0)

            +

            F.when(
                F.col(
                    "FutureInvoiceExceptionPct"
                ).isNotNull(),
                1
            ).otherwise(0)
        )
    )

    # --------------------------------------------------------
    # Low future OTD
    # --------------------------------------------------------

    .withColumn(
        "FutureLowOTDFlag",
        F.when(
            (
                F.col(
                    "FutureSupplierOTDPct"
                ).isNotNull()
            )
            &
            (
                F.col(
                    "FutureSupplierOTDPct"
                )
                <= OTD_LOW_THRESHOLD
            ),
            1
        )
        .otherwise(0)
    )

    # --------------------------------------------------------
    # High future overdue exposure
    # --------------------------------------------------------

    .withColumn(
        "FutureHighOverdueFlag",
        F.when(
            (
                F.col(
                    "FutureOverdueOpenDeliveryExposurePct"
                ).isNotNull()
            )
            &
            (
                F.col(
                    "FutureOverdueOpenDeliveryExposurePct"
                )
                >= OVERDUE_EXPOSURE_HIGH_THRESHOLD
            ),
            1
        )
        .otherwise(0)
    )

    # --------------------------------------------------------
    # High future invoice dispute
    # --------------------------------------------------------

    .withColumn(
        "FutureHighDisputeFlag",
        F.when(
            (
                F.col(
                    "FutureInvoiceDisputePct"
                ).isNotNull()
            )
            &
            (
                F.col(
                    "FutureInvoiceDisputePct"
                )
                >= DISPUTE_HIGH_THRESHOLD
            ),
            1
        )
        .otherwise(0)
    )

    # --------------------------------------------------------
    # High future invoice exception
    # --------------------------------------------------------

    .withColumn(
        "FutureHighInvoiceExceptionFlag",
        F.when(
            (
                F.col(
                    "FutureInvoiceExceptionPct"
                ).isNotNull()
            )
            &
            (
                F.col(
                    "FutureInvoiceExceptionPct"
                )
                >= EXCEPTION_HIGH_THRESHOLD
            ),
            1
        )
        .otherwise(0)
    )

    # --------------------------------------------------------
    # Composite future risk event count
    # --------------------------------------------------------

    .withColumn(
        "FutureRiskEventCount",
        (
            F.col("FutureLowOTDFlag")
            +
            F.col("FutureHighOverdueFlag")
            +
            F.col("FutureHighDisputeFlag")
            +
            F.col(
                "FutureHighInvoiceExceptionFlag"
            )
        )
    )

    # --------------------------------------------------------
    # Final target
    # --------------------------------------------------------

    .withColumn(
        "HighRiskNextYearFlag",
        F.when(
            F.col(
                "FutureRiskEventCount"
            ) >= 2,
            1
        )
        .otherwise(0)
    )

    # Require sufficient future evidence
    .filter(
        F.col(
            "FutureRiskObservedMetricCount"
        ) >= 2
    )
)


print(
    "Labeled training rows:",
    f"{training_labeled_df.count():,}"
)

Labeled training rows: 1,009


**Inspect risk-label distribution**

The class balance will be important when choosing the training strategy in DB_03.

In [0]:
# ============================================================
# Inspect target class balance
# ============================================================

risk_class_distribution_df = (
    training_labeled_df

    .groupBy(
        "HighRiskNextYearFlag"
    )

    .agg(
        F.count("*")
        .alias("SupplierYearCount")
    )

    .withColumn(
        "Percentage",
        (
            F.col(
                "SupplierYearCount"
            )
            /
            F.sum(
                "SupplierYearCount"
            ).over(
                Window.partitionBy()
            )
        )
        * 100.0
    )

    .orderBy(
        "HighRiskNextYearFlag"
    )
)


display(
    risk_class_distribution_df
)

HighRiskNextYearFlag,SupplierYearCount,Percentage
0,728,72.15064420218039
1,281,27.849355797819626


In [0]:
# ============================================================
# Inspect target balance by feature year
# ============================================================

risk_distribution_by_year_df = (
    training_labeled_df

    .groupBy(
        "PerformanceYear",
        "HighRiskNextYearFlag"
    )

    .agg(
        F.count("*")
        .alias("SupplierYearCount")
    )

    .withColumn(
        "YearTotalCount",
        F.sum(
            "SupplierYearCount"
        ).over(
            Window.partitionBy(
                "PerformanceYear"
            )
        )
    )

    .withColumn(
        "Percentage",
        (
            F.col("SupplierYearCount")
            / F.col("YearTotalCount")
        ) * 100.0
    )

    .orderBy(
        "PerformanceYear",
        "HighRiskNextYearFlag"
    )
)


display(
    risk_distribution_by_year_df
)

PerformanceYear,HighRiskNextYearFlag,SupplierYearCount,YearTotalCount,Percentage
2022,0,234,326,71.77914110429448
2022,1,92,326,28.22085889570552
2023,0,245,338,72.48520710059172
2023,1,93,338,27.514792899408285
2024,0,249,345,72.17391304347827
2024,1,96,345,27.82608695652174


**Create the final training feature dataset**

The future outcome columns are now deliberately removed.

DB_03 must not train using something like FutureInvoiceDisputePct.

In [0]:
# ============================================================
# Final supplier risk training dataset
# ============================================================

training_features_df = (
    training_labeled_df

    .select(
        # --------------------------------------------
        # Identifiers / lineage
        # --------------------------------------------

        "SupplierKey",
        "SupplierID",
        "SupplierName",

        F.col(
            "PerformanceYear"
        ).alias(
            "FeatureYear"
        ),

        "FutureOutcomeYear",

        # --------------------------------------------
        # Supplier attributes
        # --------------------------------------------

        "SupplierType",
        "Country",
        "Region",
        "PreferredSupplierFlag",
        "StrategicSupplierFlag",
        "ESGRating",
        "FinancialRiskScore",
        "SupplierStatus",

        # --------------------------------------------
        # Current operational performance
        # --------------------------------------------

        "ContractCompliancePct",
        "MaverickSpendPct",

        "SupplierOTDPct",
        "LateFullyReceivedPct",
        "OverdueOpenDeliveryExposurePct",

        "SupplierQualityIndexPct",
        "InvoiceDisputePct",
        "DuplicateInvoicePct",
        "ThreeWayMatchPct",
        "InvoiceExceptionPct",

        # --------------------------------------------
        # Comparable activity / spend
        # --------------------------------------------

        "AnnualizedEligibleSpendEUR",
        "AnnualizedFullyReceivedPOItemCount",
        "AnnualizedInvoiceCount",
        "AnnualizedDisputedInvoiceCount",
        "AnnualizedOverdueOpenDeliveryCount",

        # --------------------------------------------
        # Historical trends
        # --------------------------------------------

        "PriorYearSpendEUR",
        "YoYSpendChangePct",

        "PriorYearOTDPct",
        "YoYOTDChangePp",

        "PriorYearInvoiceDisputePct",
        "YoYInvoiceDisputeChangePp",

        "PriorYearMaverickSpendPct",
        "YoYMaverickSpendChangePp",

        # --------------------------------------------
        # Spend stability / importance
        # --------------------------------------------

        "Rolling3YObservationCount",
        "Rolling3YAverageSpendEUR",
        "Rolling3YSpendStdDevEUR",
        "SpendVolatilityPct",
        "SupplierSpendSharePct",
        "LogAnnualizedEligibleSpendEUR",

        # --------------------------------------------
        # Observation quality
        # --------------------------------------------

        "RiskMetricObservedCount",
        "RiskMetricCoveragePct",

        # --------------------------------------------
        # Target
        # --------------------------------------------

        "HighRiskNextYearFlag"
    )

    .withColumn(
        "SourceAsOfDate",
        F.lit(
            AS_OF_DATE
        ).cast("date")
    )

    .withColumn(
        "FeatureEngineeringTimestampUTC",
        F.current_timestamp()
    )
)


print(
    "Final training rows:",
    f"{training_features_df.count():,}"
)

print(
    "Final training columns:",
    len(
        training_features_df.columns
    )
)

Final training rows: 1,009
Final training columns: 47


**Create the 2026 scoring population**

In [0]:
# ============================================================
# Build current supplier scoring population
# ============================================================

scoring_features_df = (
    supplier_feature_df

    .filter(
        F.col("PerformanceYear")
        == SCORING_YEAR
    )

    .select(
        "SupplierKey",
        "SupplierID",
        "SupplierName",

        F.col(
            "PerformanceYear"
        ).alias(
            "FeatureYear"
        ),

        # Supplier attributes
        "SupplierType",
        "Country",
        "Region",
        "PreferredSupplierFlag",
        "StrategicSupplierFlag",
        "ESGRating",
        "FinancialRiskScore",
        "SupplierStatus",

        # Operational performance
        "ContractCompliancePct",
        "MaverickSpendPct",

        "SupplierOTDPct",
        "LateFullyReceivedPct",
        "OverdueOpenDeliveryExposurePct",

        "SupplierQualityIndexPct",
        "InvoiceDisputePct",
        "DuplicateInvoicePct",
        "ThreeWayMatchPct",
        "InvoiceExceptionPct",

        # Comparable activity / spend
        "AnnualizedEligibleSpendEUR",
        "AnnualizedFullyReceivedPOItemCount",
        "AnnualizedInvoiceCount",
        "AnnualizedDisputedInvoiceCount",
        "AnnualizedOverdueOpenDeliveryCount",

        # Trends
        "PriorYearSpendEUR",
        "YoYSpendChangePct",

        "PriorYearOTDPct",
        "YoYOTDChangePp",

        "PriorYearInvoiceDisputePct",
        "YoYInvoiceDisputeChangePp",

        "PriorYearMaverickSpendPct",
        "YoYMaverickSpendChangePp",

        # Spend behavior
        "Rolling3YObservationCount",
        "Rolling3YAverageSpendEUR",
        "Rolling3YSpendStdDevEUR",
        "SpendVolatilityPct",
        "SupplierSpendSharePct",
        "LogAnnualizedEligibleSpendEUR",

        # Observation quality
        "RiskMetricObservedCount",
        "RiskMetricCoveragePct"
    )

    .withColumn(
        "SourceAsOfDate",
        F.lit(
            AS_OF_DATE
        ).cast("date")
    )

    .withColumn(
        "FeatureEngineeringTimestampUTC",
        F.current_timestamp()
    )
)


print(
    "2026 scoring suppliers:",
    f"{scoring_features_df.count():,}"
)

2026 scoring suppliers: 356


**Quality gates**

In [0]:
# ============================================================
# ML feature engineering quality gates
# ============================================================

training_row_count = (
    training_features_df.count()
)

scoring_row_count = (
    scoring_features_df.count()
)


training_duplicate_count = (
    training_features_df

    .groupBy(
        "SupplierID",
        "FeatureYear"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


scoring_duplicate_count = (
    scoring_features_df

    .groupBy(
        "SupplierID",
        "FeatureYear"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


training_null_target_count = (
    training_features_df

    .filter(
        F.col(
            "HighRiskNextYearFlag"
        ).isNull()
    )

    .count()
)


training_class_count = (
    training_features_df

    .select(
        "HighRiskNextYearFlag"
    )

    .distinct()

    .count()
)


invalid_training_year_count = (
    training_features_df

    .filter(
        F.col(
            "FeatureYear"
        )
        > MAX_FEATURE_YEAR_FOR_TRAINING
    )

    .count()
)


invalid_scoring_year_count = (
    scoring_features_df

    .filter(
        F.col(
            "FeatureYear"
        )
        != SCORING_YEAR
    )

    .count()
)


quality_checks = [
    (
        "Training dataset contains rows",
        training_row_count > 0
    ),
    (
        "Scoring dataset contains rows",
        scoring_row_count > 0
    ),
    (
        "Training grain is unique",
        training_duplicate_count == 0
    ),
    (
        "Scoring grain is unique",
        scoring_duplicate_count == 0
    ),
    (
        "Training target contains no nulls",
        training_null_target_count == 0
    ),
    (
        "Training target contains both classes",
        training_class_count == 2
    ),
    (
        "No incomplete future year used for training",
        invalid_training_year_count == 0
    ),
    (
        "Scoring population is 2026 only",
        invalid_scoring_year_count == 0
    )
]


failed_checks = [
    name
    for name, passed
    in quality_checks
    if not passed
]


for check_name, passed in quality_checks:
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{check_name}"
    )


if failed_checks:
    raise ValueError(
        "DB_02 quality gate FAILED: "
        + "; ".join(
            failed_checks
        )
    )


print(
    "\nDB_02 QUALITY GATE PASSED."
)

PASS | Training dataset contains rows
PASS | Scoring dataset contains rows
PASS | Training grain is unique
PASS | Scoring grain is unique
PASS | Training target contains no nulls
PASS | Training target contains both classes
PASS | No incomplete future year used for training
PASS | Scoring population is 2026 only

DB_02 QUALITY GATE PASSED.


**Feature null profile**

We should not automatically replace legitimate nulls such as OTD for a supplier with no completed deliveries.

DB_03 will handle missing values inside the ML preprocessing pipeline.

In [0]:
# ============================================================
# Training feature null profile
# ============================================================

non_feature_columns = {
    "SupplierKey",
    "SupplierID",
    "SupplierName",
    "FeatureYear",
    "FutureOutcomeYear",
    "HighRiskNextYearFlag",
    "SourceAsOfDate",
    "FeatureEngineeringTimestampUTC"
}


feature_columns = [
    column_name
    for column_name
    in training_features_df.columns
    if column_name
    not in non_feature_columns
]


null_aggregation = [
    F.sum(
        F.when(
            F.col(column_name).isNull(),
            1
        ).otherwise(0)
    ).alias(
        column_name
    )
    for column_name
    in feature_columns
]


null_counts = (
    training_features_df

    .agg(
        *null_aggregation
    )

    .collect()[0]

    .asDict()
)


null_profile_rows = [
    (
        column_name,
        int(
            null_counts[
                column_name
            ]
        ),
        float(
            null_counts[
                column_name
            ]
            / training_row_count
            * 100.0
        )
    )
    for column_name
    in feature_columns
]


null_profile_df = (
    spark.createDataFrame(
        null_profile_rows,
        [
            "FeatureName",
            "NullCount",
            "NullPct"
        ]
    )

    .orderBy(
        F.desc(
            "NullPct"
        )
    )
)


display(
    null_profile_df
)

FeatureName,NullCount,NullPct
YoYInvoiceDisputeChangePp,441,43.70664023785927
PriorYearInvoiceDisputePct,422,41.823587710604556
YoYMaverickSpendChangePp,418,41.427155599603566
YoYOTDChangePp,415,41.12983151635283
PriorYearOTDPct,405,40.13875123885035
PriorYearMaverickSpendPct,399,39.54410307234886
YoYSpendChangePct,399,39.54410307234886
PriorYearSpendEUR,391,38.75123885034688
Rolling3YSpendStdDevEUR,374,37.06640237859267
SpendVolatilityPct,374,37.06640237859267


**Persist the ML feature datasets to OneLake**

They go under Files, not Tables.

In [0]:
# ============================================================
# Persist supplier-risk ML feature datasets
# ============================================================

(
    training_features_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        TRAINING_FEATURES_PATH
    )
)


(
    scoring_features_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        SCORING_FEATURES_PATH
    )
)


print(
    "Training features written successfully."
)

print(
    "Scoring features written successfully."
)

Training features written successfully.
Scoring features written successfully.


**Persist the target definition**

In [0]:
# ============================================================
# Persist supplier risk label definition
# ============================================================

label_threshold_rows = [
    (
        "SupplierOTDPct",
        "LOW",
        0.25,
        float(
            OTD_LOW_THRESHOLD
        )
    ),
    (
        "OverdueOpenDeliveryExposurePct",
        "HIGH",
        0.75,
        float(
            OVERDUE_EXPOSURE_HIGH_THRESHOLD
        )
    ),
    (
        "InvoiceDisputePct",
        "HIGH",
        0.75,
        float(
            DISPUTE_HIGH_THRESHOLD
        )
    ),
    (
        "InvoiceExceptionPct",
        "HIGH",
        0.75,
        float(
            EXCEPTION_HIGH_THRESHOLD
        )
    )
]


label_threshold_schema = StructType([
    StructField(
        "MetricName",
        StringType(),
        False
    ),
    StructField(
        "AdverseDirection",
        StringType(),
        False
    ),
    StructField(
        "Quantile",
        DoubleType(),
        False
    ),
    StructField(
        "ThresholdValue",
        DoubleType(),
        False
    )
])


label_thresholds_df = (
    spark.createDataFrame(
        label_threshold_rows,
        schema=label_threshold_schema
    )

    .withColumn(
        "MinimumAdverseConditionsForHighRisk",
        F.lit(2)
    )

    .withColumn(
        "LatestCompleteOutcomeYear",
        F.lit(
            LATEST_COMPLETE_YEAR
        )
    )

    .withColumn(
        "SourceAsOfDate",
        F.lit(
            AS_OF_DATE
        ).cast("date")
    )

    .withColumn(
        "CreatedTimestampUTC",
        F.current_timestamp()
    )
)


(
    label_thresholds_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        LABEL_THRESHOLDS_PATH
    )
)


display(
    label_thresholds_df
)

MetricName,AdverseDirection,Quantile,ThresholdValue,MinimumAdverseConditionsForHighRisk,LatestCompleteOutcomeYear,SourceAsOfDate,CreatedTimestampUTC
SupplierOTDPct,LOW,0.25,82.54,2,2025,2026-07-31,2026-08-12T18:19:53.284703Z
OverdueOpenDeliveryExposurePct,HIGH,0.75,5.952380952380952,2,2025,2026-07-31,2026-08-12T18:19:53.284703Z
InvoiceDisputePct,HIGH,0.75,3.7037037037037033,2,2025,2026-07-31,2026-08-12T18:19:53.284703Z
InvoiceExceptionPct,HIGH,0.75,9.68,2,2025,2026-07-31,2026-08-12T18:19:53.284703Z


**Read-back validation**

In [0]:
# ============================================================
# Validate persisted ML feature datasets
# ============================================================

training_features_validation_df = (
    spark.read
    .format("delta")
    .load(
        TRAINING_FEATURES_PATH
    )
)


scoring_features_validation_df = (
    spark.read
    .format("delta")
    .load(
        SCORING_FEATURES_PATH
    )
)


persisted_training_count = (
    training_features_validation_df.count()
)

persisted_scoring_count = (
    scoring_features_validation_df.count()
)


if (
    persisted_training_count
    != training_row_count
):
    raise ValueError(
        "Training feature write validation failed."
    )


if (
    persisted_scoring_count
    != scoring_row_count
):
    raise ValueError(
        "Scoring feature write validation failed."
    )


print(
    "DB_02 persistence validation PASSED."
)

print(
    "Training rows:",
    f"{persisted_training_count:,}"
)

print(
    "Scoring rows:",
    f"{persisted_scoring_count:,}"
)

print(
    "\nDB_02 SUPPLIER RISK FEATURE ENGINEERING PASSED."
)

DB_02 persistence validation PASSED.
Training rows: 1,009
Scoring rows: 356

DB_02 SUPPLIER RISK FEATURE ENGINEERING PASSED.


**Modeling Decisions**

- 2026 is not used as a training outcome. It is YTD and would distort the target.
- Spend is annualized for 2026 scoring. That makes the current partial year more comparable with prior complete years.
- The target represents future adverse performance. We are not simply teaching a model to reproduce a current risk formula.
- ESG and FinancialRiskScore remain predictors rather than target components. This allows us to test whether they actually help predict future operational problems.
- Country and Region remain categorical supplier features. We do not currently have a genuine external country-risk score in the synthetic source, so we should not fabricate one merely to satisfy the SAD wording.
- Future outcome columns are removed before persistence. That protects DB_03 from target leakage.

For DB_03, we will  use the persisted training dataset to build the full modeling pipeline with categorical encoding, missing-value treatment, temporal train/test split, baseline model, candidate models, class imbalance handling, ROC-AUC/PR-AUC/precision/recall/F1, feature importance, MLflow experiment tracking, model selection, and scoring of the 2026 population.